# Fair ML Baselines: No Cell-Line ID Leakage

This notebook runs ML baselines (XGBoost, Ridge) using **biological features only** — no cell-line IDs.

Features: `drug_embed(768) + rna_embed(256) + cancer_type_onehot(30) + tissue_onehot(26)` = 1080 dims

**Why this matters:** Cell-line IDs (learnable embeddings in DL, one-hot in ML) are always zero/garbage
for unseen test cell lines in cell-line-aware CV splits. This notebook removes that unfair feature.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path
import json

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy import stats

try:
    import xgboost as xgb
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'xgboost'])
    import xgboost as xgb

from gastro_transformer.data import (
    DrugEmbeddingDataset,
    IC50Dataset,
)

In [ ]:
# Configuration - must match DL CV setup
N_FOLDS = 3
SEED = 42

ROOT_DIR = '../'

# Data paths
DRUG_EMBEDDINGS_CSV = ROOT_DIR + 'data/drug_embeddings.csv'
IC50_CSV = ROOT_DIR + 'data/ic50_data.csv'
CELLLINE_RNA_CSV = ROOT_DIR + 'data/processed/ccle_rna_for_ic50.csv'

OUTPUT_DIR = Path(ROOT_DIR + 'reports/cross_validation_v4')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature dimensions
DRUG_DIM = 768
RNA_DIM = 256
NUM_CANCER_TYPES = 30
NUM_TISSUE_TYPES = 26

print(f"N folds: {N_FOLDS}")
print(f"Seed: {SEED}")
print(f"Feature dims: {DRUG_DIM} (drug) + {RNA_DIM} (RNA) + {NUM_CANCER_TYPES} (cancer) + {NUM_TISSUE_TYPES} (tissue) = {DRUG_DIM + RNA_DIM + NUM_CANCER_TYPES + NUM_TISSUE_TYPES}")

In [ ]:
# Load data
print("Loading drug embeddings...")
drug_embeddings = DrugEmbeddingDataset(
    csv_path=DRUG_EMBEDDINGS_CSV,
    drug_dim=768
)

print("\nLoading IC50 dataset...")
ic50_dataset = IC50Dataset(
    ic50_csv_path=IC50_CSV,
    drug_embeddings=drug_embeddings,
    rna_csv_path=CELLLINE_RNA_CSV,
    rna_dim=256,
    add_tissue_ids=True
)

print(f"\nTotal IC50 samples: {len(ic50_dataset)}")
print(f"Unique cell-lines: {ic50_dataset.num_celllines}")

In [ ]:
# Extract FAIR features (no cell-line IDs!)
print("Extracting fair features (no cell-line IDs)...")

n_samples = len(ic50_dataset)
total_dim = DRUG_DIM + RNA_DIM + NUM_CANCER_TYPES + NUM_TISSUE_TYPES

features = np.zeros((n_samples, total_dim), dtype=np.float32)
targets = np.zeros(n_samples, dtype=np.float32)
rna_available = np.zeros(n_samples, dtype=bool)

for i in range(n_samples):
    item = ic50_dataset[i]
    offset = 0
    
    # Drug embedding: 768 dims
    drug_embed = item['drug_embed'].numpy()
    features[i, offset:offset + DRUG_DIM] = drug_embed
    offset += DRUG_DIM
    
    # RNA embedding: 256 dims (zeros if unavailable)
    if 'rna_embed' in item:
        rna_embed = item['rna_embed'].numpy()
        features[i, offset:offset + RNA_DIM] = rna_embed
        if 'rna_available' in item:
            rna_available[i] = item['rna_available'].item()
        else:
            rna_available[i] = np.any(rna_embed != 0)
    offset += RNA_DIM
    
    # Cancer type one-hot: 30 dims
    if 'cancer_type_id' in item:
        ct = item['cancer_type_id'].item()
        if 0 <= ct < NUM_CANCER_TYPES:
            features[i, offset + ct] = 1.0
    offset += NUM_CANCER_TYPES
    
    # Tissue type one-hot: 26 dims
    if 'tissue_id' in item:
        tt = item['tissue_id'].item()
        if 0 <= tt < NUM_TISSUE_TYPES:
            features[i, offset + tt] = 1.0
    offset += NUM_TISSUE_TYPES
    
    targets[i] = item['ic50'].item()

print(f"Feature matrix shape: {features.shape}")
print(f"Target shape: {targets.shape}")
print(f"RNA available: {rna_available.sum()}/{n_samples} ({rna_available.mean()*100:.1f}%)")
print(f"\nNO cell-line ID features — fair comparison with feature-based DL model")

In [ ]:
# Create cell-line aware CV splits (SAME logic as DL models and original ML notebook)
print("Creating cell-line aware CV splits...")

rng = np.random.default_rng(SEED)

unique_celllines = sorted(ic50_dataset.cellline_to_idx.keys())
rng.shuffle(unique_celllines)

n_celllines = len(unique_celllines)
fold_size = n_celllines // N_FOLDS

cellline_folds = []
for fold_idx in range(N_FOLDS):
    start = fold_idx * fold_size
    end = start + fold_size if fold_idx < N_FOLDS - 1 else n_celllines
    fold_celllines = set(unique_celllines[start:end])
    cellline_folds.append(fold_celllines)

print(f"Created {N_FOLDS} folds with {fold_size}-{fold_size+1} cell-lines each")

sample_fold = np.zeros(n_samples, dtype=int)
for i in range(n_samples):
    cl_id = ic50_dataset.cellline_ids[i]
    for fold_idx, fold_cl in enumerate(cellline_folds):
        if cl_id in fold_cl:
            sample_fold[i] = fold_idx
            break

print(f"Sample distribution per fold: {np.bincount(sample_fold)}")

In [ ]:
def compute_metrics(y_true, y_pred):
    """Compute regression metrics."""
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    pearson_r, _ = stats.pearsonr(y_true, y_pred)
    spearman_r, _ = stats.spearmanr(y_true, y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    return {
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'pearson_r': float(pearson_r),
        'spearman_r': float(spearman_r),
        'r2': float(r2)
    }

In [ ]:
# Train and evaluate ML models with CV
print("=" * 60)
print("Running FAIR ML Baselines with 3-Fold CV")
print("Features: drug_embed + rna_embed + cancer_onehot + tissue_onehot")
print("NO cell-line IDs!")
print("=" * 60)

results = {}

for model_name in ['xgboost', 'ridge']:
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    fold_metrics = []
    
    for fold_idx in range(N_FOLDS):
        print(f"\n--- Fold {fold_idx + 1}/{N_FOLDS} ---")
        
        train_mask = sample_fold != fold_idx
        test_mask = sample_fold == fold_idx
        
        X_train = features[train_mask]
        y_train = targets[train_mask]
        X_test = features[test_mask]
        y_test = targets[test_mask]
        
        print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        if model_name == 'xgboost':
            model = xgb.XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=SEED,
                n_jobs=-1,
                verbosity=0
            )
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
        else:
            model = Ridge(alpha=1.0, random_state=SEED)
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
        
        metrics = compute_metrics(y_test, y_pred)
        fold_metrics.append(metrics)
        
        print(f"Fold {fold_idx + 1} - R²: {metrics['r2']:.4f}, Pearson: {metrics['pearson_r']:.4f}, Spearman: {metrics['spearman_r']:.4f}")
    
    avg_metrics = {}
    for key in fold_metrics[0].keys():
        values = [m[key] for m in fold_metrics]
        avg_metrics[key] = float(np.mean(values))
        avg_metrics[f'{key}_std'] = float(np.std(values))
    
    for key in ['r2', 'pearson_r', 'spearman_r', 'rmse', 'mae']:
        values = [m[key] for m in fold_metrics]
        sem = np.std(values) / np.sqrt(N_FOLDS)
        avg_metrics[f'{key}_ci95'] = float(1.96 * sem)
    
    results[model_name] = {
        'fold_metrics': fold_metrics,
        'average_metrics': avg_metrics
    }
    
    print(f"\n{model_name.upper()} Average:")
    print(f"  R²: {avg_metrics['r2']:.4f} ± {avg_metrics['r2_std']:.4f}")
    print(f"  Pearson R: {avg_metrics['pearson_r']:.4f} ± {avg_metrics['pearson_r_std']:.4f}")
    print(f"  Spearman R: {avg_metrics['spearman_r']:.4f} ± {avg_metrics['spearman_r_std']:.4f}")

In [ ]:
# Save results
results['config'] = {
    'n_folds': N_FOLDS,
    'seed': SEED,
    'feature_dims': {
        'drug_embedding': DRUG_DIM,
        'rna_embedding': RNA_DIM,
        'cancer_type_onehot': NUM_CANCER_TYPES,
        'tissue_type_onehot': NUM_TISSUE_TYPES,
        'total': DRUG_DIM + RNA_DIM + NUM_CANCER_TYPES + NUM_TISSUE_TYPES
    },
    'description': 'Fair ML baselines — NO cell-line ID features. Uses biological features only.'
}

output_path = OUTPUT_DIR / 'ml_baselines_fair_cv_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_path}")

In [ ]:
# Compare with original ML baselines (with cell-line IDs)
original_path = OUTPUT_DIR / 'ml_baselines_cv_results.json'
if original_path.exists():
    with open(original_path, 'r') as f:
        original_results = json.load(f)
    
    print("\n" + "=" * 80)
    print("COMPARISON: Fair vs Original ML Baselines")
    print("=" * 80)
    
    print(f"\n{'Model':<30} {'R²':>8} {'Pearson':>10} {'Spearman':>10}")
    print("-" * 58)
    
    for model_name in ['xgboost', 'ridge']:
        # Original (with cell-line IDs)
        orig_m = original_results[model_name]['average_metrics']
        print(f"{model_name + ' (+ cellline ID)':<30} {orig_m['r2']:>8.4f} {orig_m['pearson_r']:>10.4f} {orig_m['spearman_r']:>10.4f}")
        
        # Fair (no cell-line IDs)
        fair_m = results[model_name]['average_metrics']
        print(f"{model_name + ' (fair, no ID)':<30} {fair_m['r2']:>8.4f} {fair_m['pearson_r']:>10.4f} {fair_m['spearman_r']:>10.4f}")
        
        diff = fair_m['r2'] - orig_m['r2']
        print(f"{'  -> delta R²':<30} {diff:>+8.4f}")
        print()
else:
    print("Original ML baselines not found, skipping comparison.")

In [ ]:
# Summary
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

for model_name in ['xgboost', 'ridge']:
    m = results[model_name]['average_metrics']
    print(f"\n{model_name.upper()} (fair):")
    print(f"  R² = {m['r2']:.4f} ± {m['r2_std']:.4f}")
    print(f"  Pearson R = {m['pearson_r']:.4f} ± {m['pearson_r_std']:.4f}")
    print(f"  Spearman R = {m['spearman_r']:.4f} ± {m['spearman_r_std']:.4f}")
    print(f"  RMSE = {m['rmse']:.4f} ± {m['rmse_std']:.4f}")
    print(f"  MAE = {m['mae']:.4f} ± {m['mae_std']:.4f}")

print("\nThese are the baselines to compare against the feature-based DL model")
print("(ModalitySlotQFormer with use_feature_cellline_encoder=True)")